# MINE: Regime II, OpenFold + ESM-1b (Representational Compression)

Separate notebook for OpenFold/ESM-1b evaluation. Saves results to shared Drive directory.

In [ ]:
import os
import numpy as np

PHASE = 'full'  # 'quick' or 'full'

OUTPUT_BASE = './results/mine_v2_three_regimes/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR   = OUTPUT_BASE + 'cache'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

CONFIG = {
    'quick': {
        'n_sequences': 100,
        'seq_lengths': [100, 200],
        'mine_epochs': 200,
        'mine_batch_size': 64,
        'mine_lr': 1e-4,
        'mine_hidden': [256, 128],
        'n_runs': 3,
    },
    'full': {
        'n_sequences': 200,
        'seq_lengths': [100, 200, 400],
        'mine_epochs': 500,
        'mine_batch_size': 64,
        'mine_lr': 1e-4,
        'mine_hidden': [256, 128],
        'n_runs': 5,
    },
}

RANDOM_SEED = 320
config = CONFIG[PHASE]

print(f"Phase: {PHASE.upper()}")
print(f"Sequences per class: {config['n_sequences']}")
print(f"Sequence lengths: {config['seq_lengths']}")
print(f"MINE epochs: {config['mine_epochs']}")
print(f"MINE independent runs: {config['n_runs']}")
print(f"Results: {RESULTS_DIR}")

## Install Dependencies

In [ ]:
OPENFOLD_DIR = os.environ.get('OPENFOLD_DIR', '/content/openfold')
WEIGHTS_DIR = os.environ.get('OF_WEIGHTS_DIR', './results/openfold_weights')

import os, sys

print("Installing base dependencies...")
!pip install -q matplotlib seaborn pandas scipy scikit-learn numpy

print("Installing ESM (for OpenFold pipeline)...")
!pip install -q fair-esm

print("Installing OpenFold dependencies...")
!pip install -q biopython ml-collections modelcif awscli

# Install OpenFold
try:
    import openfold
except ImportError:
    !pip install -q "openfold @ git+https://github.com/aqlaboratory/openfold.git" 2>/dev/null
    try:
        import openfold
    except ImportError:
        if not os.path.exists(OPENFOLD_DIR):
            !git clone --depth 1 -q https://github.com/aqlaboratory/openfold.git {OPENFOLD_DIR}
        sys.path.insert(0, OPENFOLD_DIR)

# Download OpenFold SoloSeq weights
OF_WEIGHTS_DIR = WEIGHTS_DIR
OF_WEIGHTS_FILE = 'seq_model_esm1b_ptm.pt'
OF_WEIGHTS_PATH = f'{OF_WEIGHTS_DIR}/{OF_WEIGHTS_FILE}'
if not os.path.exists(OF_WEIGHTS_PATH) or os.path.getsize(OF_WEIGHTS_PATH) < 50_000_000:
    os.makedirs(OF_WEIGHTS_DIR, exist_ok=True)
    import subprocess, shutil
    dl_dir = f'{OF_WEIGHTS_DIR}/openfold_soloseq_params'
    os.makedirs(dl_dir, exist_ok=True)
    subprocess.run([
        'aws', 's3', 'cp', '--no-sign-request', '--region', 'us-east-1',
        's3://openfold/openfold_soloseq_params/', dl_dir, '--recursive'
    ], check=False)
    for root, _, files in os.walk(OF_WEIGHTS_DIR):
        for f in files:
            if f == OF_WEIGHTS_FILE:
                src = os.path.join(root, f)
                if os.path.abspath(src) != os.path.abspath(OF_WEIGHTS_PATH):
                    shutil.copy2(src, OF_WEIGHTS_PATH)
                break

import torch, numpy as np
print(f"\ntorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Done!")

## MINE Core Implementation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class MINEStatisticsNetwork(nn.Module):
    # Statistics network T_theta for MINE.
    # Takes concatenated (x, x_hat) as input and outputs a scalar.
    def __init__(self, input_dim, hidden_dims=[256, 128]):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(prev_dim, h), nn.ReLU(), nn.Dropout(0.1)])
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def mine_estimate(T_net, joint_batch, marginal_batch):
    # Compute the Donsker-Varadhan lower bound on MI.
    # I(X;X_hat) >= E_joint[T(x,x_hat)] - log(E_marginal[exp(T(x,x_hat))])
    t_joint = T_net(joint_batch)
    t_marginal = T_net(marginal_batch)
    first_term = t_joint.mean()
    second_term = torch.logsumexp(t_marginal, dim=0) - np.log(t_marginal.shape[0])
    mi_lb = first_term - second_term
    return mi_lb


def run_mine(X_bio, X_emb, n_epochs, batch_size, lr, hidden_dims, seed,
             device='cuda', verbose=True):
    # Run a single MINE estimation.
    torch.manual_seed(seed)
    np.random.seed(seed)

    N = X_bio.shape[0]
    d_bio = X_bio.shape[1]
    d_emb = X_emb.shape[1]
    input_dim = d_bio + d_emb

    X_bio_t = torch.FloatTensor(
        (X_bio - X_bio.mean(0)) / (X_bio.std(0) + 1e-8)
    ).to(device)
    X_emb_t = torch.FloatTensor(
        (X_emb - X_emb.mean(0)) / (X_emb.std(0) + 1e-8)
    ).to(device)

    T_net = MINEStatisticsNetwork(input_dim, hidden_dims).to(device)
    optimizer = optim.Adam(T_net.parameters(), lr=lr)

    mi_estimates = []

    for epoch in range(n_epochs):
        perm = torch.randperm(N)
        marginal_perm = torch.randperm(N)
        epoch_mi = []

        for i in range(0, N - batch_size + 1, batch_size):
            idx = perm[i:i + batch_size]
            marginal_idx = marginal_perm[i:i + batch_size]

            x_bio_batch = X_bio_t[idx]
            x_emb_batch = X_emb_t[idx]

            joint = torch.cat([x_bio_batch, x_emb_batch], dim=1)
            x_emb_marginal = X_emb_t[marginal_idx]
            marginal = torch.cat([x_bio_batch, x_emb_marginal], dim=1)

            mi_lb = mine_estimate(T_net, joint, marginal)
            loss = -mi_lb
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(T_net.parameters(), max_norm=5.0)
            optimizer.step()

            epoch_mi.append(mi_lb.item())

        mean_mi = np.mean(epoch_mi)
        mi_estimates.append(mean_mi)

        if verbose and (epoch + 1) % 50 == 0:
            print(f"  Epoch {epoch+1}/{n_epochs}: MI_lb = {mean_mi:.4f}")

    tail = max(1, len(mi_estimates) // 10)
    final_mi = np.mean(mi_estimates[-tail:])
    return mi_estimates, final_mi


def run_mine_with_ci(X_bio, X_emb, config, device='cuda', label=''):
    # Run MINE multiple times and compute confidence intervals.
    all_traces = []
    all_finals = []

    for run_idx in range(config['n_runs']):
        seed = RANDOM_SEED + run_idx * 100
        if label:
            print(f"\n  [{label}] Run {run_idx+1}/{config['n_runs']} (seed={seed})")
        traces, final_mi = run_mine(
            X_bio, X_emb,
            n_epochs=config['mine_epochs'],
            batch_size=config['mine_batch_size'],
            lr=config['mine_lr'],
            hidden_dims=config['mine_hidden'],
            seed=seed,
            device=device,
            verbose=(run_idx == 0),
        )
        all_traces.append(traces)
        all_finals.append(final_mi)
        print(f"    Final MI = {final_mi:.4f}")

    mean_mi = np.mean(all_finals)
    std_mi = np.std(all_finals)
    print(f"\n  [{label}] MI = {mean_mi:.4f} +/- {std_mi:.4f}")
    return all_traces, all_finals, mean_mi, std_mi


print("MINE implementation ready")
print(f"  Architecture: input -> {config['mine_hidden']} -> 1")
print(f"  Training: {config['mine_epochs']} epochs, lr={config['mine_lr']}")

## MINE Sanity Check

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('='*70)
print('MINE SANITY CHECK: Correlated Gaussians with known MI')
print('='*70)

sanity_results = []
for rho in [0.0, 0.3, 0.6, 0.9]:
    N_sanity = 2000
    rng_sc = np.random.default_rng(RANDOM_SEED)
    z1 = rng_sc.standard_normal(N_sanity)
    z2 = rng_sc.standard_normal(N_sanity)
    x_sanity = z1.reshape(-1, 1).astype(np.float32)
    y_sanity = (rho * z1 + np.sqrt(1 - rho**2) * z2).reshape(-1, 1).astype(np.float32)

    mi_true = -0.5 * np.log(1 - rho**2) if rho != 0 else 0.0

    traces_sc, mi_est = run_mine(
        x_sanity, y_sanity, n_epochs=300, batch_size=64,
        lr=1e-4, hidden_dims=[128, 64], seed=RANDOM_SEED,
        device=device, verbose=False,
    )
    sanity_results.append({'rho': rho, 'mi_true': mi_true, 'mi_est': mi_est})
    tol_ok = abs(mi_est - mi_true) < max(0.15, 0.3 * mi_true)
    status = 'PASS' if tol_ok else 'FAIL'
    print(f'  rho={rho:.1f}: MI_true={mi_true:.4f}, MI_est={mi_est:.4f}  [{status}]')

print('\nIf estimates track the true values, MINE is calibrated.')
print('Proceed to real model evaluation.\n')

## Biological Ground Truth (Proteins)

In [ ]:
import urllib.request

AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')

ECOLI_URL = 'https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=(organism_id:83333)+AND+(reviewed:true)&size=500'
HUMAN_URL = 'https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=(organism_id:9606)+AND+(reviewed:true)&size=500'


def download_proteins(url, n_sequences, min_length, seed, cache_tag):
    '''Download and cache protein sequences from UniProt.'''
    cache_file = f'{CACHE_DIR}/{cache_tag}_{n_sequences}.txt'
    if os.path.exists(cache_file):
        with open(cache_file) as f:
            seqs = [line.strip() for line in f if line.strip()]
        print(f'  Loaded {len(seqs)} cached ({cache_tag})')
        return seqs

    print(f'  Downloading from UniProt ({cache_tag})...')
    req = urllib.request.Request(url, headers={'User-Agent': 'Python/mine_v2'})
    with urllib.request.urlopen(req) as response:
        raw = response.read().decode('ascii')

    proteins, current = [], []
    for line in raw.split('\n'):
        if line.startswith('>'):
            if current:
                proteins.append(''.join(current))
            current = []
        elif line.strip():
            current.append(line.strip().upper())
    if current:
        proteins.append(''.join(current))

    valid_aa = set(AMINO_ACIDS)
    valid = [p for p in proteins if len(p) >= min_length
             and all(c in valid_aa for c in p[:min_length])]

    rng = np.random.default_rng(seed)
    if len(valid) > n_sequences:
        idx = rng.choice(len(valid), n_sequences, replace=False)
        valid = [valid[i] for i in idx]

    seqs = [p[:min_length] for p in valid[:n_sequences]]
    with open(cache_file, 'w') as f:
        f.write('\n'.join(seqs))
    print(f'  Got {len(seqs)} sequences ({cache_tag})')
    return seqs


def compute_aa_features(sequences):
    '''Compute amino acid composition + biophysical features.'''
    features = []
    for seq in sequences:
        L = len(seq)
        aa_freq = np.array([seq.count(aa) / L for aa in AMINO_ACIDS])
        len_feat = np.array([L / 1000.0])
        charge = (seq.count('K') + seq.count('R') - seq.count('D') - seq.count('E')) / L
        hydro_aa = {'A':1.8,'C':2.5,'D':-3.5,'E':-3.5,'F':2.8,'G':-0.4,
                    'H':-3.2,'I':4.5,'K':-3.9,'L':3.8,'M':1.9,'N':-3.5,
                    'P':-1.6,'Q':-3.5,'R':-4.5,'S':-0.8,'T':-0.7,'V':4.2,
                    'W':-0.9,'Y':-1.3}
        hydro = np.mean([hydro_aa.get(aa, 0) for aa in seq])
        extra = np.array([charge, hydro])
        features.append(np.concatenate([aa_freq, len_feat, extra]))
    return np.array(features, dtype=np.float32)


def pad_protein(signal_seq, target_length, rng):
    pad_needed = target_length - len(signal_seq)
    if pad_needed <= 0:
        return signal_seq[:target_length]
    pad = ''.join(rng.choice(list(AMINO_ACIDS), size=pad_needed))
    return signal_seq + pad


def cleanup_gpu():
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


# Download proteins
print("Downloading protein sequences...")
SIGNAL_LENGTH = 50
ecoli_seqs = download_proteins(ECOLI_URL, config['n_sequences'], SIGNAL_LENGTH, RANDOM_SEED, 'mine_ecoli')
human_seqs = download_proteins(HUMAN_URL, config['n_sequences'], SIGNAL_LENGTH, RANDOM_SEED, 'mine_human')

all_protein_signals = ecoli_seqs + human_seqs
protein_labels = np.array([0]*len(ecoli_seqs) + [1]*len(human_seqs))

X_bio_protein = compute_aa_features(all_protein_signals)
species_onehot = np.zeros((len(protein_labels), 2), dtype=np.float32)
species_onehot[np.arange(len(protein_labels)), protein_labels] = 1.0
X_bio_protein = np.concatenate([X_bio_protein, species_onehot], axis=1)

print(f"\nProtein ground truth: {X_bio_protein.shape} (AA comp + charge + hydro + species)")
print(f"Total sequences: {len(all_protein_signals)} ({len(ecoli_seqs)} E.coli + {len(human_seqs)} Human)")

## PCA + MI Ceiling

In [ ]:
from sklearn.decomposition import PCA

MINE_PCA_DIM = 50

def pca_reduce_embeddings(X_emb, n_components=MINE_PCA_DIM, seed=RANDOM_SEED):
    '''Reduce embedding dimensionality via PCA for stable MINE estimation.'''
    if X_emb.shape[1] <= n_components:
        print(f"    PCA: skipped (dim={X_emb.shape[1]} <= {n_components})")
        return X_emb, None

    actual_components = min(n_components, X_emb.shape[0], X_emb.shape[1])
    pca = PCA(n_components=actual_components, random_state=seed)
    X_reduced = pca.fit_transform(X_emb).astype(np.float32)
    var_explained = pca.explained_variance_ratio_.sum()
    print(f"    PCA: {X_emb.shape[1]}d -> {actual_components}d "
          f"({var_explained:.1%} variance retained)")
    return X_reduced, pca


def calibrate_mi_ceiling(X_bio, config, device='cuda', label='', noise_scale=0.1):
    '''Estimate MI ceiling by running MINE with X_bio as both X and X_hat.'''
    print(f"\n  Calibrating MI ceiling for {label}...")
    rng = np.random.default_rng(RANDOM_SEED)
    X_perfect = X_bio + noise_scale * rng.standard_normal(X_bio.shape).astype(np.float32)

    traces, finals, mean_mi, std_mi = run_mine_with_ci(
        X_bio, X_perfect, config, device=device,
        label=f'Ceiling-{label}'
    )
    print(f"  MI Ceiling ({label}): {mean_mi:.4f} +/- {std_mi:.4f} nats")
    return mean_mi, std_mi


print(f"PCA reduction: all embeddings reduced to {MINE_PCA_DIM}d before MINE")
print("MI ceiling calibration: run MINE(X_bio, X_bio + noise) for each distribution")

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('='*70)
print('MI CEILING CALIBRATION: Protein')
print('='*70)

mi_ceiling_protein, mi_ceiling_protein_std = calibrate_mi_ceiling(
    X_bio_protein, config, device=device, label='Protein'
)

print(f'\nCeiling: Protein = {mi_ceiling_protein:.4f} +/- {mi_ceiling_protein_std:.4f} nats')

## Random Baseline

In [ ]:
import time

print('\n' + '='*70)
print('RANDOM BASELINE: Protein Distribution')
print('='*70)

rng_rand = np.random.default_rng(RANDOM_SEED)

X_emb_rand_protein = rng_rand.standard_normal(
    (X_bio_protein.shape[0], MINE_PCA_DIM)
).astype(np.float32)

t0 = time.time()
traces, finals, rand_mi_protein, rand_std_protein = run_mine_with_ci(
    X_bio_protein, X_emb_rand_protein, config, device=device,
    label='Random-Protein')

noise_floor_protein = rand_mi_protein + 2 * rand_std_protein
print(f'\nNoise floor (protein): {noise_floor_protein:.4f} nats')

## OpenFold Embedding Function

In [ ]:
import torch, gc
import torch.nn as nn

# Mock flash_attn BEFORE any openfold/esm imports.
import sys, types, importlib.util

for _mod_name in ['flash_attn', 'flash_attn.flash_attn_interface',
                  'flash_attn.flash_attn_triton_amd',
                  'flash_attn.bert_padding',
                  'flash_attn_2_cuda']:
    _m = types.ModuleType(_mod_name)
    _m.__spec__ = importlib.util.spec_from_loader(_mod_name, loader=None)
    _m.__path__ = []
    _m.__file__ = ''
    _m.__package__ = _mod_name.rsplit('.', 1)[0] if '.' in _mod_name else _mod_name
    _m.flash_attn_func = None
    _m.flash_attn_varlen_func = None
    _m.flash_attn_with_kvcache = None
    _m.flash_attn_varlen_kvpacked_func = None
    _m.unpad_input = None
    _m.pad_input = None
    sys.modules[_mod_name] = _m

print("Mocked flash_attn to bypass ABI mismatch")


def make_openfold_embed_fn(device='cuda', signal_length=None):
    import esm as esm_module
    import torch.nn.functional as F
    from openfold.config import model_config
    from openfold.model.model import AlphaFold

    RESTYPES = 'ARNDCQEGHILKMFPSTWYV'
    AA_TO_IDX = {aa: i for i, aa in enumerate(RESTYPES)}

    esm_model, esm_alphabet = esm_module.pretrained.esm1b_t33_650M_UR50S()
    esm_model = esm_model.eval().to(device)
    batch_converter = esm_alphabet.get_batch_converter()

    cfg = model_config("seq_model_esm1b_ptm", train=False)
    cfg.model.template.enabled = False
    cfg.globals.use_flash = False
    cfg.globals.use_deepspeed_evo_attention = False
    of_model = AlphaFold(cfg)
    state = torch.load(OF_WEIGHTS_PATH, map_location='cpu')
    of_model.load_state_dict(state, strict=False)
    of_model = of_model.eval().to(device)

    _captured = {}
    _evoformer_count = [0]
    _fallback_count = [0]

    def _hook(module, inp, out):
        m, z, s = out
        _captured['s'] = s.detach()
    of_model.evoformer.register_forward_hook(_hook)

    @torch.no_grad()
    def embed(sequences):
        '''Returns (evoformer_embs, esm1b_embs) from same forward pass.'''
        all_evo_embs = []
        all_esm_embs = []
        for idx, seq in enumerate(sequences):
            seq = seq[:1022]
            L = len(seq)

            _, _, tokens = batch_converter([("p", seq)])
            tokens = tokens.to(device)
            esm_out = esm_model(tokens, repr_layers=[33])
            seq_emb = esm_out['representations'][33][0, 1:L+1, :]

            esm_pool_len = min(signal_length, seq_emb.shape[0]) if signal_length else seq_emb.shape[0]
            esm_pooled = seq_emb[:esm_pool_len].mean(0).cpu().numpy()
            all_esm_embs.append(esm_pooled)

            NUM_ITERS = 1
            aatype = torch.tensor(
                [AA_TO_IDX.get(aa, 20) for aa in seq],
                dtype=torch.long, device=device
            )
            target_feat = F.one_hot(aatype.clamp(max=21), num_classes=22).float()
            residue_index = torch.arange(L, dtype=torch.long, device=device)
            seq_mask = torch.ones(L, device=device, dtype=torch.float32)
            msa_mask = torch.ones(1, L, device=device, dtype=torch.float32)
            msa_feat = torch.zeros(1, L, 49, device=device, dtype=torch.float32)

            features = {
                'aatype':        aatype.unsqueeze(-1).expand(-1, NUM_ITERS),
                'target_feat':   target_feat.unsqueeze(-1).expand(-1, -1, NUM_ITERS),
                'residue_index': residue_index.unsqueeze(-1).expand(-1, NUM_ITERS),
                'seq_embedding': seq_emb.unsqueeze(-1).expand(-1, -1, NUM_ITERS),
                'seq_mask':      seq_mask.unsqueeze(-1).expand(-1, NUM_ITERS),
                'msa_mask':      msa_mask.unsqueeze(-1).expand(-1, -1, NUM_ITERS),
                'msa_feat':      msa_feat.unsqueeze(-1).expand(-1, -1, -1, NUM_ITERS),
            }

            _captured.clear()
            try:
                _ = of_model(features)
            except Exception as e:
                if _evoformer_count[0] + _fallback_count[0] == 0:
                    err_msg = str(e)
                    if any(k in err_msg for k in ['msa_feat', 'target_feat']) or 'evoformer' in err_msg.lower():
                        import traceback
                        print(f"\n    [DEBUG] Pre-Evoformer crash: {type(e).__name__}: {e}")
                        traceback.print_exc()

            if 's' in _captured:
                s_repr = _captured['s'].squeeze(0)
                pool_len = min(signal_length, s_repr.shape[0]) if signal_length else s_repr.shape[0]
                pooled = s_repr[:pool_len].mean(0).cpu().numpy()
                _evoformer_count[0] += 1
            else:
                pooled = esm_pooled
                _fallback_count[0] += 1
                if _fallback_count[0] <= 3:
                    print(f"\n    WARNING: Seq {idx} used ESM-1b fallback (Evoformer hook failed)")

            all_evo_embs.append(pooled)

            if (idx+1) % 10 == 0:
                print(f"    Seq {idx+1}/{len(sequences)}", end='\r')

        print()
        evo_pct = _evoformer_count[0] / max(1, _evoformer_count[0] + _fallback_count[0]) * 100
        print(f"    Evoformer: {_evoformer_count[0]} seqs ({evo_pct:.0f}%) | "
              f"ESM-1b fallback: {_fallback_count[0]} seqs")
        if _fallback_count[0] > len(sequences) * 0.5:
            print(f"    CRITICAL: >50% fallback, results may reflect ESM-1b, not OpenFold!")
        return np.stack(all_evo_embs), np.stack(all_esm_embs)

    pool_desc = f'signal_length={signal_length}' if signal_length else 'full sequence'
    print(f"OpenFold ready ({sum(p.numel() for p in of_model.parameters())/1e6:.0f}M params, pool: {pool_desc})")
    return embed, of_model

## Run MINE: Regime II (OpenFold + ESM-1b)

In [ ]:
from scipy.spatial import procrustes as scipy_procrustes
from sklearn.decomposition import PCA as skPCA
import time
import pandas as pd

mine_results = []
mine_traces = {}
procrustes_results = []

NOTEBOOK_TAG = 'openfold'

print('\n' + '='*70)
print('REGIME II: OpenFold (Representational Compression)')
print('='*70)

embed_fn_of, model_of = make_openfold_embed_fn(device=device, signal_length=SIGNAL_LENGTH)

esm1b_embeddings_by_len = {}
evoformer_embeddings_by_len = {}

for seq_len in config['seq_lengths']:
    print(f'\n--- OpenFold L={seq_len} (pooling first {SIGNAL_LENGTH} tokens) ---')
    rng_pad = np.random.default_rng(RANDOM_SEED)
    padded = [pad_protein(s, seq_len, rng_pad) for s in all_protein_signals]

    cache_evo = f'{CACHE_DIR}/openfold_mine_sigpool_L{seq_len}.npy'
    cache_esm = f'{CACHE_DIR}/esm1b_raw_mine_sigpool_L{seq_len}.npy'

    if os.path.exists(cache_evo) and os.path.exists(cache_esm):
        X_emb = np.load(cache_evo)
        X_esm = np.load(cache_esm)
        print(f'  Loaded cached: Evoformer={X_emb.shape}, ESM-1b={X_esm.shape}')
    else:
        print(f'  Computing embeddings ({len(padded)} seqs)...')
        X_emb, X_esm = embed_fn_of(padded)
        X_emb = np.nan_to_num(X_emb, nan=0.0, posinf=1e6, neginf=-1e6)
        X_esm = np.nan_to_num(X_esm, nan=0.0, posinf=1e6, neginf=-1e6)
        np.save(cache_evo, X_emb)
        np.save(cache_esm, X_esm)
        print(f'  Saved: Evoformer={X_emb.shape}, ESM-1b={X_esm.shape}')

    esm1b_embeddings_by_len[seq_len] = X_esm
    evoformer_embeddings_by_len[seq_len] = X_emb

    # MINE: Evoformer
    t0 = time.time()
    X_emb_pca, _ = pca_reduce_embeddings(X_emb)

    traces, finals, mean_mi, std_mi = run_mine_with_ci(
        X_bio_protein, X_emb_pca, config, device=device, label=f'OpenFold-L{seq_len}')

    mine_results.append({
        'model': 'OpenFold', 'regime': 'II', 'seq_length': seq_len,
        'mi_mean': mean_mi, 'mi_std': std_mi, 'mi_runs': finals,
        'time_min': (time.time()-t0)/60,
        'ground_truth': 'protein', 'mi_ceiling': mi_ceiling_protein,
        'mi_normalized': mean_mi / max(mi_ceiling_protein, 1e-8),
    })
    mine_traces[f'OpenFold-L{seq_len}'] = traces

del model_of
cleanup_gpu()

# ESM-1b baseline + Procrustes
print('\n' + '='*70)
print('ESM-1b BASELINE + PROCRUSTES')
print('='*70)

for seq_len in config['seq_lengths']:
    print(f'\n--- ESM-1b Raw L={seq_len} ---')
    X_esm = esm1b_embeddings_by_len[seq_len]
    X_evo = evoformer_embeddings_by_len[seq_len]

    t0 = time.time()
    X_esm_pca, _ = pca_reduce_embeddings(X_esm)

    traces, finals, mean_mi, std_mi = run_mine_with_ci(
        X_bio_protein, X_esm_pca, config, device=device, label=f'ESM1b_Raw-L{seq_len}')

    mine_results.append({
        'model': 'ESM1b_Raw', 'regime': 'II_baseline', 'seq_length': seq_len,
        'mi_mean': mean_mi, 'mi_std': std_mi, 'mi_runs': finals,
        'time_min': (time.time()-t0)/60,
        'ground_truth': 'protein', 'mi_ceiling': mi_ceiling_protein,
        'mi_normalized': mean_mi / max(mi_ceiling_protein, 1e-8),
    })
    mine_traces[f'ESM1b_Raw-L{seq_len}'] = traces

    # Procrustes alignment
    shared_dim = min(50, X_esm.shape[0] - 1)
    pca_esm = skPCA(n_components=shared_dim, random_state=RANDOM_SEED)
    pca_evo = skPCA(n_components=shared_dim, random_state=RANDOM_SEED)
    Z_esm = pca_esm.fit_transform(X_esm)
    Z_evo = pca_evo.fit_transform(X_evo)

    _, _, disparity = scipy_procrustes(Z_esm, Z_evo)

    procrustes_results.append({
        'seq_length': seq_len,
        'disparity': disparity,
        'esm_var': pca_esm.explained_variance_ratio_.sum(),
        'evo_var': pca_evo.explained_variance_ratio_.sum(),
    })
    print(f'  Procrustes disparity (ESM-1b vs Evoformer): {disparity:.6f}')

# Add random baseline
mine_results.append({
    'model': 'Random_Protein', 'regime': 'baseline', 'seq_length': 0,
    'mi_mean': rand_mi_protein, 'mi_std': rand_std_protein, 'mi_runs': [],
    'time_min': 0,
    'ground_truth': 'protein', 'mi_ceiling': mi_ceiling_protein,
    'mi_normalized': rand_mi_protein / max(mi_ceiling_protein, 1e-8),
})

## Save Results

In [ ]:
import pandas as pd
import json as _json

NOTEBOOK_TAG = 'openfold'

# Save results
results_file = f'{RESULTS_DIR}/{NOTEBOOK_TAG}_results_{PHASE}.json'
with open(results_file, 'w') as f:
    clean_results = []
    for r in mine_results:
        cr = {}
        for k, v in r.items():
            if isinstance(v, np.floating):
                cr[k] = float(v)
            elif isinstance(v, np.integer):
                cr[k] = int(v)
            elif isinstance(v, np.ndarray):
                cr[k] = v.tolist()
            elif isinstance(v, list):
                cr[k] = [float(x) if isinstance(x, np.floating) else x for x in v]
            else:
                cr[k] = v
        clean_results.append(cr)
    _json.dump(clean_results, f, indent=2)

# Save traces
traces_file = f'{RESULTS_DIR}/{NOTEBOOK_TAG}_traces_{PHASE}.json'
clean_traces = {}
for k, v in mine_traces.items():
    clean_traces[k] = [list(map(float, t)) for t in v]
with open(traces_file, 'w') as f:
    _json.dump(clean_traces, f)

# Save Procrustes separately
proc_file = f'{RESULTS_DIR}/{NOTEBOOK_TAG}_procrustes_{PHASE}.json'
with open(proc_file, 'w') as f:
    _json.dump(procrustes_results, f, indent=2)

print(f'Saved results: {results_file}')
print(f'Saved traces: {traces_file}')
print(f'Saved Procrustes: {proc_file}')

df = pd.DataFrame(clean_results)
print('\n' + df.to_string(index=False))